# CloudShop – chạy trên Kaggle (máy phụ)

Notebook này dùng Kaggle làm **máy sinh tải và thu số liệu**: triển khai stack (tùy chọn), bắn tải bằng Locust vào Application Load Balancer, thu số liệu CloudWatch, vẽ biểu đồ và xóa stack.

**Chuẩn bị:**
1. Settings (bên phải) → **Internet: On**.
2. Add-ons → **Secrets** → thêm 2 secret `AWS_ACCESS_KEY_ID` và `AWS_SECRET_ACCESS_KEY` (khóa của IAM user, không dùng root) rồi tick chọn cho notebook này.
3. Sửa ô cấu hình bên dưới.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL   = "https://github.com/<ten-github>/cloudshop.git"   # repo công khai của nhóm
REGION     = "ap-southeast-1"
STACK      = "cloudshop"
USERS      = 150      # số người dùng ảo
SPAWN_RATE = 5        # số người dùng thêm mỗi giây
DURATION   = "15m"    # thời gian bắn tải
USER_CLASS = "HeavyShopper"   # hoặc "Shopper" (chỉ duyệt web, không có tác vụ nặng)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["AWS_ACCESS_KEY_ID"] = s.get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = s.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_REGION"] = os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["STACK_NAME"] = STACK
!pip install -q boto3 locust matplotlib
!rm -rf /kaggle/working/cloudshop && git clone -q $REPO_URL /kaggle/working/cloudshop
%cd /kaggle/working/cloudshop

## (Tùy chọn) Triển khai stack từ Kaggle
Bỏ qua ô này nếu đã triển khai bằng VS Code trên máy chính. Cần điền mật khẩu bên dưới (chỉ chữ và số).

In [ ]:
# os.environ.update(GitRepoUrl=REPO_URL, DBPassword="DoiMatKhauNay123",
#                   AppSecretKey="DoiChuoiBiMatNay2026abc", AdminPassword="Admin@2026")
# !python scripts/deploy.py --env /dev/null
!python scripts/deploy.py --env /dev/null --status

## Lấy URL website từ outputs của stack

In [ ]:
import boto3
outs = {o["OutputKey"]: o["OutputValue"] for o in
        boto3.client("cloudformation", region_name=REGION).describe_stacks(StackName=STACK)["Stacks"][0]["Outputs"]}
URL = outs["WebsiteURL"]; print(URL)
!curl -s $URL/health

## Bắn tải + thu số liệu song song
Thu số liệu chạy lâu hơn thời gian bắn tải 15 phút để thấy cả giai đoạn **scale in** (giảm instance) sau khi hết tải.

In [ ]:
import subprocess, re
mins = int(re.sub(r"\D", "", DURATION)) + 15
collector = subprocess.Popen(["python", "scripts/collect_metrics.py", "--stack", STACK, "--region", REGION,
                              "--minutes", str(mins)], stdout=open("collector.log", "w"), stderr=subprocess.STDOUT)
os.makedirs("results", exist_ok=True)
!locust -f loadtest/locustfile.py --host $URL --headless -u $USERS -r $SPAWN_RATE -t $DURATION --csv results/aws --html results/aws.html --only-summary $USER_CLASS

In [ ]:
collector.wait()   # chờ thu số liệu xong (thêm ~15 phút sau khi hết tải)
!tail -n 5 collector.log
import glob; csv_path = sorted(glob.glob("results/metrics_*.csv"))[-1]
!python scripts/plot_metrics.py $csv_path
!python scripts/scaling_activities.py --stack $STACK --region $REGION
from IPython.display import Image; Image(csv_path.replace(".csv", ".png"))

In [ ]:
import pandas as pd
pd.read_csv("results/aws_stats.csv")[["Name","Request Count","Failure Count","Median Response Time","95%","Average Response Time","Requests/s"]]

## Xóa stack sau khi demo xong (tránh phát sinh chi phí)

In [ ]:
# !python scripts/destroy.py --env /dev/null --yes